# Apply Excel Re-ID Mapping (auto-split by position)

Reads a manually-edited Excel ID mapping and applies it to a tracker CSV.

**Excel format:**
- Sheet `Team P0` — columns = players, rows = tracker IDs. First cell = canonical ID, rest are aliases (no frame constraint).
- Sheet `Team P1` — alternating (ID, frame) column pairs. First ID = canonical, rest are aliases with optional frame range (`4270-5750`, `4632-`, `5128`).
- A cell like `1 REF` / `2 P` / `GK1` means **tracker id 1/2 when detected as referee / player / goalkeeper** (class filter). `25 PT1` → the `PT1` note is ignored.

**Rules:**
1. Data before `MIN_FRAME` is never touched.
2. A tracker ID that is the *first value* (canonical) of any column is **never overwritten** for the matching class — clean player tracks stay put.
3. Non-canonical fragment IDs are merged into their column's canonical.
4. When a fragment is claimed by several columns, each frame's detection is assigned to whichever candidate's box is **nearest in pixels** at that frame.
5. After remapping, duplicate `(frame, id)` boxes are deduped (highest-conf kept) so every ID appears once per frame.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
TRACKS_CSV  = "per_frame_tracks_HILHAZ_half1.csv"   # input tracker CSV
EXCEL_FILE  = "half1_edited_v2.xlsx"                # your Excel mapping
OUTPUT_CSV  = "per_frame_tracks_half1_reid.csv"     # output

MIN_FRAME   = 4270   # ignore all frames below this
TOL         = 10     # ±frames tolerance around frame-range endpoints
POS_W       = 12     # ±frames window for the positional nearest-box tiebreak

In [ ]:
import pandas as pd, numpy as np, re
from collections import defaultdict

CLS_LETTER = {1: 'GK', 2: 'P', 3: 'REF'}   # class_id -> letter
PEOPLE     = (1, 2, 3)

def parse_cell(v):
    if pd.isna(v): return None
    s = str(v).strip()
    if not s or s.lower() == 'nan': return None
    m = re.match(r'^GK\s*(\d+)', s, re.I)
    if m: return (int(m.group(1)), 'GK')
    m = re.match(r'^(\d+)\s*(.*?)$', s)
    if not m: return None
    tid = int(m.group(1)); ann = m.group(2).strip().upper()
    if 'REF' in ann: return (tid, 'REF')
    if ann == 'P':   return (tid, 'P')
    if 'GK' in ann:  return (tid, 'GK')
    return (tid, '')          # plain / PTx -> matches any class

def frange(v):
    if pd.isna(v): return (None, None)
    s = str(v).strip()
    if not s or s.lower() == 'nan': return (None, None)
    if '-' in s:
        a, b = (s.split('-') + [''])[:2]
        return (int(a) if a.strip() else None, int(b) if b.strip() else None)
    try: f = int(s); return (f, f)
    except: return (None, None)

# ── canonicals + alias rules ──────────────────────────────────────────────────
canon_quals = defaultdict(set)   # id -> {class quals that are canonical}
alias_rules = []                 # {id, cls, target, fs, fe, origin}

p0 = pd.read_excel(EXCEL_FILE, sheet_name='Team P0', header=None)
for c in range(p0.shape[1]):
    cells = [parse_cell(p0.iloc[r, c]) for r in range(1, p0.shape[0])]
    cells = [x for x in cells if x]
    if not cells: continue
    cid, ccls = cells[0]; canon_quals[cid].add(ccls)
    for (sid, scls) in cells[1:]:
        alias_rules.append(dict(id=sid, cls=scls, target=cid, fs=None, fe=None,
                                origin=f'P0PL{c+1}'))

p1 = pd.read_excel(EXCEL_FILE, sheet_name='Team P1', header=None)
for i in range(0, p1.shape[1], 2):
    if i + 1 >= p1.shape[1]: break
    rows = []
    for r in range(1, p1.shape[0]):
        idc = parse_cell(p1.iloc[r, i]); fr = frange(p1.iloc[r, i + 1])
        if idc: rows.append((idc, fr))
    if not rows: continue
    (cid, ccls), _ = rows[0]; canon_quals[cid].add(ccls)
    for (sid, scls), (fs, fe) in rows[1:]:
        alias_rules.append(dict(id=sid, cls=scls, target=cid, fs=fs, fe=fe,
                                origin=f'P1PL{i//2+1}'))

def is_protected(tid, cls_id):
    q = canon_quals.get(tid)
    return bool(q) and (('' in q) or (CLS_LETTER[cls_id] in q))

def cls_match(rule_cls, cls_id):
    return rule_cls == '' or rule_cls == CLS_LETTER[cls_id]

def frame_in(rule, f):
    if rule['fs'] is not None and f < rule['fs'] - TOL: return False
    if rule['fe'] is not None and f > rule['fe'] + TOL: return False
    return True

# ── load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv(TRACKS_CSV, encoding='utf-8', encoding_errors='replace', low_memory=False)
orig = df['display_track_id'].copy()
df['_cx'] = (df['x1'] + df['x2']) / 2
df['_cy'] = (df['y1'] + df['y2']) / 2
ppl_mask = df['class_id'].isin(PEOPLE)

pos_by_id = {int(t): g[['frame', '_cx', '_cy']].sort_values('frame').to_numpy()
             for t, g in df[ppl_mask].groupby('display_track_id')}

def nearest_target(cands, f, cx, cy):
    best, bestd = None, 1e18
    for T in cands:
        arr = pos_by_id.get(T)
        if arr is None: continue
        sel = np.abs(arr[:, 0] - f) <= POS_W
        if not sel.any():
            sel = np.abs(arr[:, 0] - f) <= 50
            if not sel.any(): continue
        d = np.min(np.hypot(arr[sel, 1] - cx, arr[sel, 2] - cy))
        if d < bestd: bestd, best = d, T
    return best

# ── apply remap ───────────────────────────────────────────────────────────────
work = df[ppl_mask & (df['frame'] >= MIN_FRAME)]
new_ids = df['display_track_id'].copy()
n_prot = n_single = n_split = n_none = 0
split_detail = defaultdict(int)

for idx, r in work.iterrows():
    tid, cid, f = int(r.display_track_id), int(r.class_id), int(r.frame)
    if is_protected(tid, cid):
        n_prot += 1; continue
    cands = []
    for rule in alias_rules:
        if rule['id'] == tid and cls_match(rule['cls'], cid) and frame_in(rule, f):
            if rule['target'] not in cands: cands.append(rule['target'])
    if not cands:
        n_none += 1; continue
    if len(cands) == 1:
        new_ids.at[idx] = cands[0]; n_single += 1
    else:
        T = nearest_target(cands, f, r._cx, r._cy) or cands[0]
        new_ids.at[idx] = T; n_split += 1; split_detail[(tid, T)] += 1

df['display_track_id'] = new_ids
df.drop(columns=['_cx', '_cy'], inplace=True)
assert ((df['display_track_id'] != orig) & (df['frame'] < MIN_FRAME)).sum() == 0

print(f"protected(kept): {n_prot}   single merges: {n_single}   "
      f"positional splits: {n_split}   no-rule(kept): {n_none}")
for (s, t), n in sorted(split_detail.items()):
    print(f"   split {s} -> {t}: {n} rows")

# ── per-frame dedupe (frames>=MIN_FRAME, people): keep highest-conf box ────────
keep_lo  = df[(df['frame'] < MIN_FRAME) | (~df['class_id'].isin(PEOPLE))]
dedup_hi = df[(df['frame'] >= MIN_FRAME) & df['class_id'].isin(PEOPLE)]
before = len(dedup_hi)
dedup_hi = (dedup_hi.sort_values('conf', ascending=False)
                    .drop_duplicates(['frame', 'display_track_id'], keep='first'))
out = (pd.concat([keep_lo, dedup_hi])
         .sort_values(['frame', 'display_track_id']).reset_index(drop=True))
print(f"\ndeduped {before - len(dedup_hi)} duplicate (frame,id) boxes")

out.to_csv(OUTPUT_CSV, index=False)
print(f"saved -> {OUTPUT_CSV}  ({len(out):,} rows)")
a = out[(out['frame'] >= MIN_FRAME) & out['class_id'].isin(PEOPLE)]
print(f"\nframes in window: {a['frame'].nunique()}   (each id should be <= this)")
print("ID counts (frames >= MIN_FRAME):")
print(a['display_track_id'].value_counts().sort_index().to_string())